In [0]:
# %sql

# -- Create a replica of target table to test above migration script
# CREATE TABLE default.dimrequests_test
# DEEP CLONE default.dimrequests;


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

dbutils.widgets.text("bucket", "DataStagingBucket", "S3 Bucket")
dbutils.widgets.text("env", "prod", "Environment")

# 1. Define the path to CSV file
s3_bucket = dbutils.widgets.get("bucket")
env = dbutils.widgets.get("env")

# -------------------- Edit these for csv path, target table name and schema -------------------

file_path = f"s3a://{s3_bucket}/edw_exports/{env}/dimRequests__202603021245.csv"

target_table_name = "default.dimrequests"

# schema mapping CSV's columns
csv_schema = StructType([
    StructField("foirequestid", IntegerType(), True),
    StructField("sourceoftruth", StringType(), True),
    StructField("sourceoftruthuniqueid", IntegerType(), True),
    StructField("visualrequestfilenumber", StringType(), True),
    StructField("cactive", StringType(), True),
    StructField("createddate", TimestampType(), True),
    StructField("modifieddate", TimestampType(), True)
])

# 2. Read the CSV into a DataFrame
csv_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(csv_schema)
    .load(file_path)
)

print(f"Number of rows in CSV DataFrame: {csv_df.count()}")

# 3. Upsert (Merge) into the target Delta table
if spark.catalog.tableExists(target_table_name):
    delta_table = DeltaTable.forName(spark, target_table_name)
    
    # merge condition
    merge_condition = """
        target.foirequestid = source.foirequestid AND 
        target.sourceoftruth = source.sourceoftruth
    """
    
    (delta_table.alias("target")
        .merge(
            csv_df.alias("source"),
            merge_condition 
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Migration complete: CSV data upserted.")

else:
    (csv_df.write
        .format("delta")
        .mode("overwrite") 
        .saveAsTable(target_table_name)
    )
    print("Migration complete: New Delta table created from CSV.")